In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for visuals
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12

# Resolve path to sql_connector
BASE_DIR = r'C:\Users\Ayush\Git Repo\Stack-Over-Flow-Survey-Data-Engineering-Project\Data Warehouse'
sys.path.append(os.path.join(BASE_DIR, 'Silver Layer', 'utils'))
from sql_connector import SQLConnector

db = SQLConnector("Stack_Overflow_Survey")
db.connect()

In [ ]:
# Query median salary trends globally and in top countries
salary_query = """
SELECT 
    f.SurveyYear,
    d.Country,
    f.ConvertedCompYearly,
    f.YearsCodePro
FROM Snowflake.Fact_Survey_Core f
JOIN Snowflake.Dim_Demographics d ON f.Dim_DemographicsID = d.Dim_DemographicsID
WHERE f.ConvertedCompYearly IS NOT NULL 
  AND f.ConvertedCompYearly > 5000 
  AND f.ConvertedCompYearly < 500000;
"""
print("Fetching salary data from database...")
df_salary = db.read_query(salary_query)
df_salary['Year'] = pd.to_datetime(df_salary['SurveyYear']).dt.year
print(f"Loaded {len(df_salary)} rows.")

In [ ]:
# Filter for top countries
top_countries = ['United States of America', 'India', 'United Kingdom of Great Britain and Northern Ireland', 'Germany', 'Canada']
df_top_salary = df_salary[df_salary['Country'].isin(top_countries)]

# Plot Median Salary over time by country
median_salaries = df_top_salary.groupby(['Year', 'Country'])['ConvertedCompYearly'].median().reset_index()

plt.figure(figsize=(12, 6))
sns.lineplot(data=median_salaries, x='Year', y='ConvertedCompYearly', hue='Country', marker='o', linewidth=2.5)
plt.title('Median Developer Salary Trends by Country (2021-2025)')
plt.xlabel('Survey Year')
plt.ylabel('Median Salary (USD)')
plt.xticks([2021, 2022, 2023, 2024, 2025])
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('salary_trends_by_country.png', dpi=300)
plt.show()

In [ ]:
# Filter to 2025 data to see salary vs experience status
df_2025 = df_top_salary[df_top_salary['Year'] == 2025].copy()

def bucket_exp(exp):
    try:
        val = int(exp)
        if val < 2: return '0-1 years'
        elif val < 5: return '2-4 years'
        elif val < 10: return '5-9 years'
        elif val < 15: return '10-14 years'
        else: return '15+ years'
    except:
        return 'Unknown'

df_2025['Experience_Group'] = df_2025['YearsCodePro'].apply(bucket_exp)
df_2025 = df_2025[df_2025['Experience_Group'] != 'Unknown']

order = ['0-1 years', '2-4 years', '5-9 years', '10-14 years', '15+ years']
plt.figure(figsize=(12, 6))
sns.boxplot(data=df_2025[df_2025['Country'] == 'United States of America'], x='Experience_Group', y='ConvertedCompYearly', order=order, palette='Blues')
plt.title('US Developer Salaries by Professional Experience (2025)')
plt.xlabel('Years of Professional Experience')
plt.ylabel('Salary (USD)')
plt.tight_layout()
plt.savefig('salary_by_experience.png', dpi=300)
plt.show()

In [ ]:
# Query top programming languages using Bridge view
lang_query = """
SELECT 
    YEAR(f.SurveyYear) AS Year,
    b.LanguageHaveWorkedWith_Clean AS Language,
    COUNT(DISTINCT f.ResponseKey) AS Count
FROM Snowflake.Fact_Survey_Core f
JOIN Snowflake.Bridge_LanguageHaveWorkedWith_Clean b ON f.ResponseKey = b.ResponseKey
GROUP BY YEAR(f.SurveyYear), b.LanguageHaveWorkedWith_Clean;
"""
print("Fetching language adoption data...")
df_lang = db.read_query(lang_query)
print(f"Loaded {len(df_lang)} language rows.")

In [ ]:
# Calculate percentage share for each language per year
year_totals = df_lang.groupby('Year')['Count'].sum().reset_index().rename(columns={'Count': 'TotalCount'})
df_lang_share = df_lang.merge(year_totals, on='Year')
df_lang_share['Percentage'] = (df_lang_share['Count'] / df_lang_share['TotalCount']) * 100

# Get top 8 languages in 2025 to track their history
top_langs_2025 = df_lang[df_lang['Year'] == 2025].nlargest(8, 'Count')['Language'].tolist()
df_top_langs = df_lang_share[df_lang_share['Language'].isin(top_langs_2025)]

# Plot trends of top languages
plt.figure(figsize=(12, 6))
sns.lineplot(data=df_top_langs, x='Year', y='Percentage', hue='Language', marker='s', linewidth=2)
plt.title('Top Programming Languages Trend Share (2021-2025)')
plt.xlabel('Year')
plt.ylabel('Share of Developer Responses (%)')
plt.xticks([2021, 2022, 2023, 2024, 2025])
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('language_trends.png', dpi=300)
plt.show()

In [ ]:
# Query AI sentiments from Dim_AIOpinions (2023-2025)
ai_query = """
SELECT 
    YEAR(f.SurveyYear) AS Year,
    o.AIBen AS AI_Benefit_Sentiment,
    COUNT(DISTINCT f.ResponseKey) AS Count
FROM Snowflake.Fact_Survey_Core f
JOIN Snowflake.Dim_AICentral c ON f.Dim_AICentralID = c.Dim_AICentralID
JOIN Snowflake.Dim_AIOpinions o ON c.Dim_AIOpinionsID = o.Dim_AIOpinionsID
WHERE o.AIBen IS NOT NULL AND o.AIBen <> ''
GROUP BY YEAR(f.SurveyYear), o.AIBen;
"""
print("Fetching AI sentiment data...")
df_ai = db.read_query(ai_query)
print(f"Loaded {len(df_ai)} AI sentiment rows.")

In [ ]:
# Plot AI benefits sentiment distribution
plt.figure(figsize=(12, 6))
sns.barplot(data=df_ai, x='AI_Benefit_Sentiment', y='Count', hue='Year', palette='viridis')
plt.title('Developer Sentiment on the Benefits of AI (2023-2025)')
plt.xlabel('AI Sentiment')
plt.ylabel('Respondent Count')
plt.xticks(rotation=15)
plt.legend(title='Year')
plt.tight_layout()
plt.savefig('ai_sentiment_trends.png', dpi=300)
plt.show()

In [ ]:
# Query remote vs in-person work over time
remote_query = """
SELECT 
    YEAR(f.SurveyYear) AS Year,
    cat.RemoteWork,
    COUNT(DISTINCT f.ResponseKey) AS Count
FROM Snowflake.Fact_Survey_Core f
JOIN Snowflake.Dim_Employment cat ON f.Dim_EmploymentID = cat.Dim_EmploymentID
WHERE cat.RemoteWork IS NOT NULL AND cat.RemoteWork <> '' AND cat.RemoteWork <> 'Unknown'
GROUP BY YEAR(f.SurveyYear), cat.RemoteWork;
"""
print("Fetching remote work trends...")
df_remote = db.read_query(remote_query)
print(f"Loaded {len(df_remote)} remote work rows.")

In [ ]:
# Pivot to make stacked area plot
df_pivot = df_remote.pivot(index='Year', columns='RemoteWork', values='Count').fillna(0)
# Normalize to percentages
df_pivot_pct = df_pivot.div(df_pivot.sum(axis=1), axis=0) * 100

plt.figure(figsize=(12, 6))
df_pivot_pct.plot(kind='area', stacked=True, alpha=0.85, colormap='Accent', ax=plt.gca())
plt.title('Workplace Dynamics Evolution (2021-2025)')
plt.xlabel('Year')
plt.ylabel('Percentage of Developers (%)')
plt.xticks([2021, 2022, 2023, 2024, 2025])
plt.legend(title='Remote Status', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('remote_work_trends.png', dpi=300)
plt.show()

# Close connection
db.close()